`[1]`

# CNN de favorabilidade de fosfato — Colab

Roda no Colab os três scripts de `src/model_DL/` (plano em `plano_cnn.md`):

| Etapa | Script | Células |
|---|---|---|
| exportar patches do GEE (sem task) | `exportar_patches.py` | 6–7 |
| treinar e validar (5 folds) | `treinar_cnn.py` | 8–9 |
| teste final (uma vez) | `treinar_cnn.py --teste` | 10 |
| mapa por janela deslizante | `inferir_cnn.py` | 11–12 |

Os scripts são os mesmos do servidor: aqui só mudam os caminhos (Drive) e os
parâmetros vêm dos formulários. Se os patches já foram exportados no servidor,
copie a pasta `patches_cnn/` para o Drive e pule as células 6–7.

**Antes:** *Ambiente de execução → Alterar o tipo → GPU* (T4 ou A100).

In [ ]:
# [2] Drive, código do projeto e pastas
from google.colab import drive
drive.mount('/content/drive')

import os, sys, shutil, subprocess
from pathlib import Path

#@markdown Código: pasta `src/` do projeto no Drive **ou** URL git (deixe vazio para usar o Drive)
SRC_DRIVE = '/content/drive/MyDrive/prospectra/modelagem_prospectra_mine/src'  #@param {type:"string"}
GIT_URL = ''  #@param {type:"string"}
#@markdown Dados e rodadas no Drive
PATCHES_DIR = '/content/drive/MyDrive/prospectra/patches_cnn'  #@param {type:"string"}
RUNS_DIR = '/content/drive/MyDrive/prospectra/runs_cnn'  #@param {type:"string"}
#@markdown Copiar os patches para o disco do Colab (leitura bem mais rápida no treino)
COPIAR_LOCAL = True  #@param {type:"boolean"}

if GIT_URL:
    if not Path('/content/repo').exists():
        subprocess.run(['git', 'clone', '-q', GIT_URL, '/content/repo'], check=True)
    SRC = Path('/content/repo/src')
else:
    SRC = Path(SRC_DRIVE)
assert (SRC / 'model_DL' / 'treinar_cnn.py').exists(), f'scripts não encontrados em {SRC}/model_DL'

Path(PATCHES_DIR).mkdir(parents=True, exist_ok=True)
Path(RUNS_DIR).mkdir(parents=True, exist_ok=True)
os.environ['PATCHES_DIR'] = PATCHES_DIR
os.environ['RUNS_DIR'] = RUNS_DIR
for p in (SRC, SRC / 'model_DL', SRC / 'modeling'):
    sys.path.insert(0, str(p))
os.chdir(SRC / 'model_DL')
print('código :', SRC)
print('patches:', PATCHES_DIR)
print('rodadas:', RUNS_DIR)

In [ ]:
# [3] pacotes (torch já vem no Colab)
%pip install -q timm earthengine-api rasterio

In [ ]:
# [4] GPU
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB')
else:
    print('Sem GPU: Ambiente de execução → Alterar o tipo → GPU')

In [ ]:
# [5] Earth Engine (exportação e inferência leem o mosaico no GEE)
import ee
PROJETO_EE = 'mapbiomas-brazil'  #@param {type:"string"}
ee.Authenticate()
ee.Initialize(project=PROJETO_EE)
print('Earth Engine OK —', PROJETO_EE)

`[5b]`

## Exportação dos patches

Mesmo fluxo de `exportar_patches.py`, sem o menu: a célula 6 sorteia os centros
e mostra as quantidades; marque `BAIXAR` para baixar os patches. Retomável —
células já gravadas são puladas. Se os patches vieram do servidor, pule.

In [ ]:
# [6] sortear centros e (opcional) baixar os patches
import json, time
import pandas as pd
import exportar_patches as ep

BAIXAR = False  #@param {type:"boolean"}
ep.PASTA_SAIDA = Path(PATCHES_DIR)

grid = ep.ler_grid()
pimg = ep.imagem_p()
df, divisao = ep.montar_amostras(grid, pimg)
df.to_csv(ep.PASTA_SAIDA / 'indice_patches.csv', index=False)
(ep.PASTA_SAIDA / 'divisao.json').write_text(json.dumps(divisao, indent=2, ensure_ascii=False))
print(pd.crosstab(df.origem_amostra, df.conjunto, margins=True).to_string())
for modelo, canais in ep.MODELOS.items():
    print(f'modelo {modelo}: {len(df)} patches × {len(canais)} canais '
          f'≈ {len(df) * len(canais) * ep.PATCH ** 2 * 2 / 1e9:.2f} GB')

if BAIXAR:
    img = ep.imagem_canais()
    for m in ep.MODELOS:
        (ep.PASTA_SAIDA / f'modelo_{m}').mkdir(parents=True, exist_ok=True)
    for cel, g in df.groupby('celula'):
        if all(ep.arquivo_modelo(m, cel).exists() for m in ep.MODELOS):
            print(f'{cel}: já existe')
            continue
        t0 = time.time()
        n = ep.baixar_celula(img, g, cel)
        print(f'{cel}: {n}/{len(g)} patches ({time.time() - t0:.0f} s)')

In [ ]:
# [7] conferir os patches: contagens por conjunto e alguns exemplos
import numpy as np
import matplotlib.pyplot as plt
import treinar_cnn as tc

MODELO_VER = 'A'  #@param ["A", "B1"]
X, meta, canais = tc.carregar(MODELO_VER, Path(PATCHES_DIR))
print(X.shape, '| canais:', canais)
print(pd.crosstab(meta.origem_amostra, [meta.conjunto, meta.fold]).to_string())

CANAL_VER = canais[0]
c = canais.index(CANAL_VER)
fig, axs = plt.subplots(3, 6, figsize=(15, 8))
for ax, origem in zip(axs, ['ocorrencia_treino', 'ambos', 'negativo']):
    idx = np.where(meta.origem_amostra == origem)[0][:6]
    for a, i in zip(ax, idx):
        a.imshow(X[i, c].astype(np.float32), cmap='magma')
        a.set_title(f'{origem}\ny={meta.y[i]:.2f}', fontsize=8); a.axis('off')
fig.suptitle(f'{MODELO_VER} · canal {CANAL_VER}'); plt.tight_layout()

`[7b]`

## Treino com validação cruzada (5 folds)

Ordem sugerida: CNN pequena a 32 px (linha de base) para A e B1, depois os
backbones a 64 px. A métrica que vale é `auc_ponto` (ocorrências reais × negativos,
célula de fora), comparada com o GTB = 0,84. Só 2 folds têm ocorrências reais.

In [ ]:
# [8] treinar (validação cruzada)
MODELO = 'A'  #@param ["A", "B1"]
BACKBONE = 'pequena'  #@param ["pequena", "regnety_016", "tf_efficientnetv2_s", "resnet50", "coatnet_0_rw_224"]
PATCH = 32  #@param [32, 64] {type:"raw"}
EPOCAS = 40  #@param {type:"integer"}
LOTE = 64  #@param {type:"integer"}
LR = 1e-3  #@param {type:"number"}
FOLDS = '0,1,2,3,4'  #@param {type:"string"}

dados = PATCHES_DIR
if COPIAR_LOCAL:                       # Drive é lento para leitura repetida
    local = Path('/content/patches_cnn')
    if not (local / f'modelo_{MODELO}').exists():
        shutil.copytree(Path(PATCHES_DIR) / f'modelo_{MODELO}', local / f'modelo_{MODELO}')
    dados = str(local)

tc.main(['--modelo', MODELO, '--backbone', BACKBONE, '--patch', str(PATCH), '--epocas', str(EPOCAS),
         '--lote', str(LOTE), '--lr', str(LR), '--folds', FOLDS, '--workers', '2',
         '--dados', dados, '--saida', RUNS_DIR])

In [ ]:
# [9] comparar as rodadas: auc_ponto por fold e fora do fold, contra o GTB
from sklearn.metrics import roc_auc_score
linhas = []
for run in sorted(Path(RUNS_DIR).glob('*/')):
    arq = run / 'metricas_folds.csv'
    if not arq.exists():
        continue
    m = pd.read_csv(arq)
    oof = pd.read_csv(run / 'oof_validacao.csv')
    geral = tc.metricas(oof, oof.p.values)
    hist = [pd.read_csv(h) for h in run.glob('historico_fold*.csv')]
    melhor_ep = np.mean([h.loc[h['auc_ponto'].idxmax() if 'auc_ponto' in h and h['auc_ponto'].notna().any()
                               else h['auc_amostra'].idxmax(), 'epoca'] for h in hist]) if hist else np.nan
    linhas.append({'rodada': run.name, 'ap_ponto_oof': geral.get('ap_ponto'),
                   'ap_ganho_oof': geral.get('ap_ganho'), 'boyce_oof': geral.get('boyce'),
                   'top10_oof': geral.get('top10'), 'auc_ponto_oof': geral.get('auc_ponto'),
                   'auc_amostra_oof': geral.get('auc_amostra'), 'brier': geral.get('brier'),
                   'auc_ponto_folds': ', '.join(f'{v:.2f}' for v in m.get('auc_ponto', pd.Series()).dropna()),
                   'melhor_epoca_media': melhor_ep})
TAB = pd.DataFrame(linhas).sort_values('ap_ponto_oof', ascending=False)
print(f'Ordenado por ap_ponto (precisão média, ocorrências × fundo). Referência GTB: auc_ponto = {tc.GTB_REF}')
TAB.round(3)

`[9b]`

## Teste final (uma vez)

Com o melhor modelo da célula 9: treina em todos os folds com número fixo de
épocas (`melhor_epoca_media`) e mede no conjunto de teste (células das
ocorrências de validação). O teste não é olhado durante o treino.

In [ ]:
# [10] teste final
MODELO_T = 'A'  #@param ["A", "B1"]
BACKBONE_T = 'regnety_016'  #@param ["pequena", "regnety_016", "tf_efficientnetv2_s", "resnet50", "coatnet_0_rw_224"]
PATCH_T = 64  #@param [32, 64] {type:"raw"}
EPOCAS_T = 20  #@param {type:"integer"}
RODAR_TESTE = False  #@param {type:"boolean"}

dados = globals().get('dados', PATCHES_DIR)     # definido na célula 8 (cópia local)
if RODAR_TESTE:
    tc.main(['--modelo', MODELO_T, '--backbone', BACKBONE_T, '--patch', str(PATCH_T),
             '--epocas', str(EPOCAS_T), '--teste', '--workers', '2',
             '--dados', dados, '--saida', RUNS_DIR])
else:
    print('Marque RODAR_TESTE para rodar (uma vez, com o modelo escolhido).')

`[10b]`

## Mapa (janela deslizante)

Usa `final.pt` da rodada se existir, senão a média dos `fold*.pt`. Comece com
passo 3 (90 m) em poucas células; passo 1 (30 m) é o mapa final.

In [ ]:
# [11] inferência
import inferir_cnn as ic
RUN = ''  #@param {type:"string"}
PASSO = 3  #@param [1, 3, 5] {type:"raw"}
CELULAS = 'L2C3,L3C3'  #@param {type:"string"}

if not RUN:
    RUN = str(sorted(Path(RUNS_DIR).glob('*/'), key=lambda p: p.stat().st_mtime)[-1])
print('rodada:', RUN)
ic.main(['--run', RUN, '--passo', str(PASSO), '--celulas', CELULAS, '--lote', '256'])

In [ ]:
# [12] ver o mapa gerado
import rasterio
from rasterio.merge import merge

pasta = Path(RUN) / f'mapas_passo{PASSO}'
arqs = sorted(pasta.glob('p_*.tif'))
fontes = [rasterio.open(a) for a in arqs]
mosaico, transform = merge(fontes, nodata=np.nan)
for f in fontes:
    f.close()
p = mosaico[0]
fig, ax = plt.subplots(figsize=(9, 9))
im = ax.imshow(p, cmap='magma', vmin=0, vmax=1,
               extent=(transform.c, transform.c + transform.a * p.shape[1],
                       transform.f + transform.e * p.shape[0], transform.f))
fig.colorbar(im, shrink=0.7, label='p(fosfato) — CNN')
ax.set_title(f'{Path(RUN).name} · passo {PASSO} · {len(arqs)} células')
print(f'p ≥ 0,5 em {100 * np.nanmean(p >= 0.5):.1f}% da área mostrada')